<a href="https://colab.research.google.com/github/sangambohare-oss/tgpcet-smart-library-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install flask pyngrok

In [3]:
import threading
from flask import Flask, render_template_string, request
from google.colab.output import eval_js

app = Flask(__name__)

# Mock Data
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available"},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued"},
    {"id": 3, "title": "Smart Grids", "category": "Electrical", "status": "Available"}
]

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Smart Library</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
</head>
<body class="bg-light">
    <nav class="navbar navbar-dark bg-primary px-4">
        <span class="navbar-brand font-weight-bold">TGPCET Smart Library</span>
    </nav>
    <div class="container mt-4">
        <div class="card p-4 mb-4 shadow-sm">
            <h4>Search Library</h4>
            <form action="/search" method="get" class="d-flex">
                <input name="q" class="form-control me-2" placeholder="Search IT, Mechanical...">
                <button type="submit" class="btn btn-success">Search</button>
            </form>
        </div>
        <div class="row">
            {% for book in book_list %}
            <div class="col-md-4 mb-3">
                <div class="card shadow-sm h-100">
                    <div class="card-body">
                        <h5>{{ book.title }}</h5>
                        <p class="text-muted">{{ book.category }}</p>
                        <span class="badge {{ 'bg-success' if book.status == 'Available' else 'bg-danger' }}">
                            {{ book.status }}
                        </span>
                    </div>
                </div>
            </div>
            {% endfor %}
        </div>
    </div>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE, book_list=books)

@app.route('/search')
def search():
    query = request.args.get('q', '').lower()
    filtered = [b for b in books if query in b['title'].lower() or query in b['category'].lower()]
    return render_template_string(HTML_TEMPLATE, book_list=filtered)

# Run Flask in a background thread so the cell doesn't "hang"
def run_app():
    app.run(port=5000)

threading.Thread(target=run_app).start()

# Generate the public link for you to click
print("Click the link below to open your TGPCET Library Website:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

 * Serving Flask app '__main__'
Click the link below to open your TGPCET Library Website:
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev


In [4]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev


In [5]:
import threading
from flask import Flask, render_template_string, request, session, redirect, url_for
from datetime import datetime, timedelta
from google.colab.output import eval_js

app = Flask(__name__)
app.secret_key = "tgpcet_secret"

# --- SMART DATABASE (In-memory for Colab) ---
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available", "count": 10, "issued_count": 55},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued", "count": 0, "issued_count": 30},
    {"id": 3, "title": "Power Systems", "category": "Electrical", "status": "Available", "count": 5, "issued_count": 12},
    {"id": 4, "title": "Digital Electronics", "category": "ETC", "status": "Available", "count": 7, "issued_count": 80}
]

# Mock user data
issued_history = [
    {"book_id": 2, "user": "Student_01", "due_date": "2023-12-20", "status": "Overdue"}
]

# --- SMART LOGIC FUNCTIONS ---
def get_fine(due_date_str):
    due_date = datetime.strptime(due_date_str, '%Y-%m-%d')
    days_late = (datetime.now() - due_date).days
    return max(0, days_late * 10)  # ₹10 per day late

def get_recommendations():
    # Suggests books with the highest issued_count (Trending)
    return sorted(books, key=lambda x: x['issued_count'], reverse=True)[:2]

# --- HTML UI (Professional Design) ---
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Smart Library</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        .hero-gradient { background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; }
        .stat-card { border-left: 5px solid #3498db; }
        .recommendation-badge { position: absolute; top: 10px; right: 10px; }
    </style>
</head>
<body class="bg-light">

    <nav class="navbar navbar-expand-lg navbar-dark bg-dark px-4">
        <a class="navbar-brand" href="/"><b>TGPCET</b> Smart Library</a>
        <div class="ms-auto">
            <span class="text-white-50 me-3">🔔 Notifications ({{ overdue_count }})</span>
            <button class="btn btn-outline-light btn-sm">Admin Login</button>
        </div>
    </nav>

    <div class="p-5 hero-gradient text-center mb-4">
        <h1 class="display-4">Smart Resource Portal</h1>
        <p class="lead">Search books, check due dates, and track your library analytics.</p>
        <div class="container w-50 mt-4">
            <form action="/search" method="get" class="input-group input-group-lg">
                <input name="q" class="form-control" placeholder="Search by Book Title, ID, or Branch...">
                <button class="btn btn-warning" type="submit">Search</button>
            </form>
        </div>
    </div>

    <div class="container">
        <div class="row mb-5 mt-n5">
            <div class="col-md-3">
                <div class="card p-3 shadow-sm stat-card">
                    <h6 class="text-muted">Total Collection</h6>
                    <h3>15,402</h3>
                </div>
            </div>
            <div class="col-md-3">
                <div class="card p-3 shadow-sm stat-card">
                    <h6 class="text-muted">Currently Issued</h6>
                    <h3>1,240</h3>
                </div>
            </div>
            <div class="col-md-3">
                <div class="card p-3 shadow-sm stat-card" style="border-color: #e74c3c;">
                    <h6 class="text-muted">Pending Fines</h6>
                    <h3>₹4,500</h3>
                </div>
            </div>
            <div class="col-md-3">
                <div class="card p-3 shadow-sm stat-card" style="border-color: #2ecc71;">
                    <h6 class="text-muted">Active Users</h6>
                    <h3>3,800</h3>
                </div>
            </div>
        </div>

        <div class="row">
            <div class="col-md-8">
                <h4 class="mb-3">📚 Library Catalog</h4>
                <div class="row">
                    {% for book in book_list %}
                    <div class="col-md-6 mb-4">
                        <div class="card h-100 shadow-sm border-0">
                            <div class="card-body">
                                <span class="badge bg-secondary mb-2">{{ book.category }}</span>
                                <h5 class="card-title">{{ book.title }}</h5>
                                <p class="small text-muted mb-1">Book ID: #LIB{{ book.id }}</p>
                                <div class="d-flex justify-content-between align-items-center mt-3">
                                    <span class="badge {{ 'bg-success' if book.status == 'Available' else 'bg-danger' }}">
                                        {{ book.status }} ({{ book.count }} left)
                                    </span>
                                    <button class="btn btn-sm btn-outline-primary">Request Book</button>
                                </div>
                            </div>
                        </div>
                    </div>
                    {% endfor %}
                </div>
            </div>

            <div class="col-md-4">
                <div class="card shadow-sm p-3 mb-4 border-0">
                    <h5>🔥 Most Issued (Smart Suggester)</h5>
                    <ul class="list-group list-group-flush mt-2">
                        {% for rec in recs %}
                        <li class="list-group-item d-flex justify-content-between align-items-center p-2">
                            {{ rec.title }}
                            <span class="badge bg-primary rounded-pill">{{ rec.issued_count }} issued</span>
                        </li>
                        {% endfor %}
                    </ul>
                </div>

                <div class="card shadow-sm p-3 border-0 bg-dark text-white">
                    <h5>🚨 Student Alert (Late Returns)</h5>
                    {% for item in history %}
                    <div class="mt-2 border-bottom border-secondary pb-2">
                        <small class="text-warning">Book: {{ item.book_id }}</small><br>
                        <small>Due: {{ item.due_date }}</small><br>
                        <small class="text-danger">Current Fine: ₹{{ item.fine }}</small>
                    </div>
                    {% endfor %}
                </div>
            </div>
        </div>
    </div>
</body>
</html>
"""

@app.route('/')
def index():
    # Process history to calculate live fines
    for item in issued_history:
        item['fine'] = get_fine(item['due_date'])

    return render_template_string(
        HTML_TEMPLATE,
        book_list=books,
        history=issued_history,
        recs=get_recommendations(),
        overdue_count=len(issued_history)
    )

@app.route('/search')
def search():
    query = request.args.get('q', '').lower()
    filtered = [b for b in books if query in b['title'].lower() or query in b['category'].lower()]
    return render_template_string(
        HTML_TEMPLATE,
        book_list=filtered,
        history=issued_history,
        recs=get_recommendations(),
        overdue_count=len(issued_history)
    )

def run_app():
    app.run(port=5000)

threading.Thread(target=run_app).start()
print("Website Live at:", eval_js("google.colab.kernel.proxyPort(5000)"))

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Website Live at: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev


In [6]:
import threading
from flask import Flask, render_template_string, request, jsonify
from datetime import datetime
from google.colab.output import eval_js

app = Flask(__name__)

# --- EXPANDED SMART DATABASE ---
# In a real app, this would be a SQL database
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available", "count": 10, "issued_count": 55},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued", "count": 0, "issued_count": 30},
    {"id": 3, "title": "Power Systems", "category": "Electrical", "status": "Available", "count": 5, "issued_count": 12}
]

# --- ADMIN UI TEMPLATE ---
ADMIN_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Admin Dashboard</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body class="bg-light">
    <nav class="navbar navbar-dark bg-dark px-4">
        <span class="navbar-brand">TGPCET <b>Admin Panel</b></span>
        <a href="/" class="btn btn-outline-light btn-sm">Switch to Student View</a>
    </nav>

    <div class="container mt-4">
        <div class="row">
            <div class="col-md-4">
                <div class="card shadow-sm p-3 mb-4">
                    <h5>Monthly Usage</h5>
                    <canvas id="adminChart"></canvas>
                </div>
                <div class="card shadow-sm p-3 bg-primary text-white">
                    <h5>Admin Quick Stats</h5>
                    <p>Total Books: {{ total_books }}</p>
                    <p>Fines Collected: ₹12,450</p>
                </div>
            </div>

            <div class="col-md-8">
                <div class="card shadow-sm p-4 mb-4">
                    <h4>Add New Book</h4>
                    <form action="/admin/add" method="POST" class="row g-3">
                        <div class="col-md-6"><input name="title" class="form-control" placeholder="Book Title" required></div>
                        <div class="col-md-4"><input name="cat" class="form-control" placeholder="Category" required></div>
                        <div class="col-md-2"><button class="btn btn-success w-100">Add</button></div>
                    </form>
                </div>

                <div class="card shadow-sm p-4">
                    <h4>Manage Inventory</h4>
                    <table class="table table-hover mt-3">
                        <thead>
                            <tr>
                                <th>ID</th><th>Title</th><th>Category</th><th>Action</th>
                            </tr>
                        </thead>
                        <tbody>
                            {% for book in book_list %}
                            <tr>
                                <td>#{{ book.id }}</td>
                                <td>{{ book.title }}</td>
                                <td>{{ book.category }}</td>
                                <td><a href="/admin/delete/{{ book.id }}" class="btn btn-sm btn-danger">Remove</a></td>
                            </tr>
                            {% endfor %}
                        </tbody>
                    </table>
                </div>
            </div>
        </div>
    </div>

    <script>
        const ctx = document.getElementById('adminChart');
        new Chart(ctx, {
            type: 'line',
            data: {
                labels: ['Week 1', 'Week 2', 'Week 3', 'Week 4'],
                datasets: [{
                    label: 'Books Issued',
                    data: [45, 82, 55, 95],
                    borderColor: '#0d6efd',
                    tension: 0.3
                }]
            }
        });
    </script>
</body>
</html>
"""

# --- ROUTES ---

@app.route('/')
def index():
    # Reuse the previous Student UI logic
    return f"<h1>Student View</h1><a href='/admin'>Go to Admin Panel</a>" # Simplified for demo

@app.route('/admin')
def admin_panel():
    return render_template_string(ADMIN_TEMPLATE, book_list=books, total_books=len(books))

@app.route('/admin/add', methods=['POST'])
def add_book():
    new_id = len(books) + 1
    new_book = {
        "id": new_id,
        "title": request.form.get('title'),
        "category": request.form.get('cat'),
        "status": "Available",
        "count": 5,
        "issued_count": 0
    }
    books.append(new_book)
    return redirect('/admin')

@app.route('/admin/delete/<int:book_id>')
def delete_book(book_id):
    global books
    books = [b for b in books if b['id'] != book_id]
    return redirect('/admin')

# --- RUNNER ---
def run_app():
    app.run(port=5000)

threading.Thread(target=run_app).start()
print("Admin Dashboard Live at:", eval_js("google.colab.kernel.proxyPort(5000)") + "/admin")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Admin Dashboard Live at: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev/admin


In [7]:
import threading
from flask import Flask, render_template_string, request, redirect
from google.colab.output import eval_js

app = Flask(__name__)

# --- SHARED DATABASE ---
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available"},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued"},
    {"id": 3, "title": "Power Systems", "category": "Electrical", "status": "Available"}
]

# --- COMBINED TEMPLATE ---
# This includes both the Search Page and the Admin Management
BASE_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Smart Library</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
</head>
<body class="bg-light">
    <nav class="navbar navbar-dark bg-dark px-4">
        <a class="navbar-brand" href="/">TGPCET Smart Library</a>
        <div class="ms-auto">
            <a href="/" class="btn btn-outline-info btn-sm">Student View</a>
            <a href="/admin" class="btn btn-outline-warning btn-sm">Admin Panel</a>
        </div>
    </nav>

    <div class="container mt-4">
        {% if view == 'admin' %}
            <h2>Admin: Manage Books</h2>
            <form action="/admin/add" method="POST" class="row g-2 mb-4">
                <div class="col-md-5"><input name="title" class="form-control" placeholder="Book Title" required></div>
                <div class="col-md-5"><input name="cat" class="form-control" placeholder="Category" required></div>
                <div class="col-md-2"><button class="btn btn-success w-100">Add</button></div>
            </form>
        {% else %}
            <div class="alert alert-primary">Welcome, Student! Search for your books below.</div>
            <form action="/search" method="get" class="d-flex mb-4">
                <input name="q" class="form-control me-2" placeholder="Search IT, Mechanical, etc...">
                <button class="btn btn-primary">Search</button>
            </form>
        {% endif %}

        <div class="row">
            {% for book in book_list %}
            <div class="col-md-4 mb-3">
                <div class="card shadow-sm h-100">
                    <div class="card-body">
                        <h5>{{ book.title }}</h5>
                        <p class="text-muted small">{{ book.category }}</p>
                        <span class="badge {{ 'bg-success' if book.status == 'Available' else 'bg-danger' }}">
                            {{ book.status }}
                        </span>
                        {% if view == 'admin' %}
                            <hr><a href="/admin/delete/{{ book.id }}" class="btn btn-sm btn-link text-danger">Remove Book</a>
                        {% endif %}
                    </div>
                </div>
            </div>
            {% endfor %}
        </div>
    </div>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(BASE_TEMPLATE, book_list=books, view='student')

@app.route('/search')
def search():
    query = request.args.get('q', '').lower()
    filtered = [b for b in books if query in b['title'].lower() or query in b['category'].lower()]
    return render_template_string(BASE_TEMPLATE, book_list=filtered, view='student')

@app.route('/admin')
def admin():
    return render_template_string(BASE_TEMPLATE, book_list=books, view='admin')

@app.route('/admin/add', methods=['POST'])
def add():
    new_book = {"id": len(books)+1, "title": request.form['title'], "category": request.form['cat'], "status": "Available"}
    books.append(new_book)
    return redirect('/admin')

@app.route('/admin/delete/<int:bid>')
def delete(bid):
    global books
    books = [b for b in books if b['id'] != bid]
    return redirect('/admin')

# --- RUNNER ---
def run():
    app.run(port=5000)

threading.Thread(target=run).start()
print("Click here for Student View:", eval_js("google.colab.kernel.proxyPort(5000)"))
print("Click here for Admin View:", eval_js("google.colab.kernel.proxyPort(5000)") + "admin")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Click here for Student View: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev
Click here for Admin View: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.devadmin


In [8]:
import threading
from flask import Flask, render_template_string, request, redirect, url_for
from datetime import datetime, timedelta
from google.colab.output import eval_js

app = Flask(__name__)

# --- SMART DATABASE (In-Memory) ---
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available", "count": 5, "is_new": True},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued", "count": 0, "is_new": False},
    {"id": 3, "title": "Smart Grids", "category": "Electrical", "status": "Available", "count": 3, "is_new": True}
]

# Student Activity Data
issued_books = [
    {"user": "Student_01", "book": "Python Programming", "due_date": "2023-12-20", "id": 101}
]

# --- SMART LOGIC ---
def calculate_fine(due_str):
    due_date = datetime.strptime(due_str, '%Y-%m-%d')
    days_late = (datetime.now() - due_date).days
    return max(0, days_late * 5) # ₹5 per day

# --- UNIFIED UI TEMPLATE ---
UI_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Smart Library</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        .navbar { background: #2c3e50; }
        .hero { background: #34495e; color: white; padding: 40px 0; border-bottom: 5px solid #f1c40f; }
        .card-new { border-top: 4px solid #f1c40f; }
        .fine-box { background: #fff5f5; border: 1px solid #feb2b2; padding: 15px; border-radius: 8px; }
    </style>
</head>
<body>
    <nav class="navbar navbar-expand-lg navbar-dark px-4">
        <a class="navbar-brand" href="/">TGPCET <b>Smart Library</b></a>
        <div class="ms-auto d-flex align-items-center">
            <span class="text-warning me-3">🔔 Notifications ({{ overdue_count }})</span>
            <a href="/admin" class="btn btn-sm btn-outline-light">Admin Portal</a>
        </div>
    </nav>

    {% if view == 'student' %}
    <div class="hero text-center">
        <h1>Welcome to the Future of Learning</h1>
        <p>TGPCET Mohgaon Campus Library</p>
        <div class="container w-50 mt-4">
            <form action="/search" method="get" class="d-flex shadow-lg">
                <input name="q" class="form-control form-control-lg border-0" placeholder="Search Title or Category...">
                <button class="btn btn-warning px-4">Search</button>
            </form>
        </div>
    </div>

    <div class="container mt-5">
        <div class="row">
            <div class="col-md-4">
                <div class="fine-box mb-4 shadow-sm">
                    <h5 class="text-danger">My Account Summary</h5>
                    <hr>
                    {% for item in my_books %}
                    <p class="mb-1"><b>{{ item.book }}</b></p>
                    <small class="text-muted">Due: {{ item.due_date }}</small><br>
                    <span class="badge bg-danger">Current Fine: ₹{{ item.fine }}</span>
                    {% endfor %}
                    <button class="btn btn-sm btn-dark w-100 mt-3">View History</button>
                </div>

                <div class="card p-3 shadow-sm bg-light">
                    <h5>Quick Links</h5>
                    <ul class="list-unstyled">
                        <li><a href="#" class="text-decoration-none">📚 Request New Book</a></li>
                        <li><a href="#" class="text-decoration-none">🔑 Change Password</a></li>
                        <li><a href="#" class="text-decoration-none">🗺️ Library Map</a></li>
                    </ul>
                </div>
            </div>

            <div class="col-md-8">
                <h4 class="mb-4">Explore Collection</h4>
                <div class="row">
                    {% for book in book_list %}
                    <div class="col-md-6 mb-4">
                        <div class="card h-100 shadow-sm border-0 {% if book.is_new %}card-new{% endif %}">
                            <div class="card-body">
                                {% if book.is_new %}<span class="badge bg-warning text-dark float-end">NEW</span>{% endif %}
                                <h5 class="card-title">{{ book.title }}</h5>
                                <p class="text-muted small">{{ book.category }}</p>
                                <div class="d-flex justify-content-between">
                                    <span class="badge {{ 'bg-success' if book.status == 'Available' else 'bg-danger' }}">
                                        {{ book.status }}
                                    </span>
                                    <small>{{ book.count }} copies left</small>
                                </div>
                            </div>
                        </div>
                    </div>
                    {% endfor %}
                </div>
            </div>
        </div>
    </div>

    {% elif view == 'admin' %}
    <div class="container mt-5">
        <div class="d-flex justify-content-between mb-4 align-items-center">
            <h2>Admin Operations Center</h2>
            <a href="/" class="btn btn-secondary btn-sm">Exit to Student View</a>
        </div>

        <div class="row">
            <div class="col-md-8">
                <div class="card p-4 shadow-sm mb-4">
                    <h5>Add to Inventory</h5>
                    <form action="/admin/add" method="POST" class="row g-2">
                        <div class="col-md-6"><input name="title" class="form-control" placeholder="Book Title" required></div>
                        <div class="col-md-4"><input name="cat" class="form-control" placeholder="Category" required></div>
                        <div class="col-md-2"><button class="btn btn-primary w-100">Add</button></div>
                    </form>
                </div>

                <div class="table-responsive">
                    <table class="table table-striped align-middle shadow-sm">
                        <thead class="table-dark">
                            <tr><th>ID</th><th>Book Title</th><th>Branch</th><th>Action</th></tr>
                        </thead>
                        <tbody>
                            {% for b in book_list %}
                            <tr>
                                <td>#{{ b.id }}</td>
                                <td><b>{{ b.title }}</b></td>
                                <td>{{ b.category }}</td>
                                <td><a href="/admin/delete/{{ b.id }}" class="btn btn-sm btn-outline-danger">Remove</a></td>
                            </tr>
                            {% endfor %}
                        </tbody>
                    </table>
                </div>
            </div>

            <div class="col-md-4">
                <div class="card p-3 shadow-sm bg-dark text-white mb-4">
                    <h5>Analytics Snapshot</h5>
                    <p class="small">Issued Books: {{ overdue_count }}</p>
                    <p class="small">Active Requests: 12</p>
                    <p class="small text-warning">Fine Forecast: ₹2,400</p>
                </div>
            </div>
        </div>
    </div>
    {% endif %}

    <footer class="mt-5 py-4 bg-light text-center border-top">
        <p class="mb-0">© 2025 TGPCET Smart Library System | Nagpur</p>
    </footer>
</body>
</html>
"""

# --- FLASK ROUTES ---

@app.route('/')
def home():
    # Calculate fines for current session
    for item in issued_books:
        item['fine'] = calculate_fine(item['due_date'])
    return render_template_string(UI_TEMPLATE, view='student', book_list=books, my_books=issued_books, overdue_count=len(issued_books))

@app.route('/admin')
def admin():
    return render_template_string(UI_TEMPLATE, view='admin', book_list=books, overdue_count=len(issued_books))

@app.route('/search')
def search():
    q = request.args.get('q', '').lower()
    res = [b for b in books if q in b['title'].lower() or q in b['category'].lower()]
    return render_template_string(UI_TEMPLATE, view='student', book_list=res, my_books=issued_books, overdue_count=len(issued_books))

@app.route('/admin/add', methods=['POST'])
def add():
    new_id = books[-1]['id'] + 1 if books else 1
    books.append({"id": new_id, "title": request.form['title'], "category": request.form['cat'], "status": "Available", "count": 5, "is_new": True})
    return redirect('/admin')

@app.route('/admin/delete/<int:bid>')
def delete(bid):
    global books
    books = [b for b in books if b['id'] != bid]
    return redirect('/admin')

# --- RUNNER ---
def start():
    app.run(port=5000)

threading.Thread(target=start).start()
print("Website is now LIVE!")
print("Student Portal:", eval_js("google.colab.kernel.proxyPort(5000)"))
print("Admin Portal:", eval_js("google.colab.kernel.proxyPort(5000)") + "admin")

Website is now LIVE!
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Student Portal: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev
Admin Portal: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.devadmin


In [9]:
import threading
from flask import Flask, render_template_string, request, redirect
from datetime import datetime
from google.colab.output import eval_js

app = Flask(__name__)

# --- SHARED DATA ---
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available", "count": 10},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued", "count": 0},
    {"id": 3, "title": "Control Systems", "category": "Electrical", "status": "Available", "count": 5}
]

# --- DARK THEME UI TEMPLATE ---
DARK_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>TGPCET Smart Library | Dark Mode</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        :root {
            --bg-dark: #0f172a;
            --card-dark: #1e293b;
            --accent: #38bdf8;
            --text-main: #f8fafc;
            --text-dim: #94a3b8;
        }
        body { background-color: var(--bg-dark); color: var(--text-main); font-family: 'Inter', sans-serif; }
        .navbar { background-color: rgba(15, 23, 42, 0.9) !important; border-bottom: 1px solid #334155; backdrop-filter: blur(10px); }
        .hero-section { padding: 80px 0; background: radial-gradient(circle at top right, #1e293b, #0f172a); border-bottom: 1px solid #334155; }
        .card { background-color: var(--card-dark); border: 1px solid #334155; border-radius: 12px; color: var(--text-main); transition: 0.3s; }
        .card:hover { border-color: var(--accent); transform: translateY(-5px); }
        .form-control { background-color: #0f172a; border: 1px solid #334155; color: white; }
        .form-control:focus { background-color: #0f172a; color: white; border-color: var(--accent); box-shadow: none; }
        .btn-primary { background-color: var(--accent); border: none; color: #0f172a; font-weight: 600; }
        .badge-available { background-color: #10b981; color: white; }
        .badge-issued { background-color: #ef4444; color: white; }
        footer { border-top: 1px solid #334155; padding: 20px 0; color: var(--text-dim); }
    </style>
</head>
<body>

    <nav class="navbar navbar-expand-lg navbar-dark sticky-top px-4">
        <a class="navbar-brand d-flex align-items-center" href="/">
            <span style="color: var(--accent); font-weight: 800; font-size: 1.5rem;">TGPCET</span>
            <span class="ms-2 fw-light">Smart Library</span>
        </a>
        <div class="ms-auto d-flex gap-2">
            <button class="btn btn-sm btn-outline-light" data-bs-toggle="modal" data-bs-target="#loginModal">Student Login</button>
            <a href="/admin" class="btn btn-sm btn-primary">Admin Panel</a>
        </div>
    </nav>

    {% if view == 'admin' %}
    <div class="container mt-5">
        <h2 class="mb-4 text-info">Admin Control Center</h2>
        <div class="card p-4 mb-4">
            <h5>Add New Volume</h5>
            <form action="/admin/add" method="POST" class="row g-3">
                <div class="col-md-5"><input name="title" class="form-control" placeholder="Book Name" required></div>
                <div class="col-md-5"><input name="cat" class="form-control" placeholder="Department" required></div>
                <div class="col-md-2"><button class="btn btn-primary w-100">Add Book</button></div>
            </form>
        </div>
    {% else %}
    <div class="hero-section text-center">
        <div class="container">
            <h1 class="display-5 fw-bold mb-3">Knowledge at your Fingertips</h1>
            <p class="text-secondary mb-5">Search the TGPCET digital archive for books, journals, and papers.</p>
            <div class="row justify-content-center">
                <div class="col-md-7">
                    <form action="/search" method="get" class="d-flex gap-2">
                        <input name="q" class="form-control form-control-lg" placeholder="Search by Title or Branch (IT, ETC, ME)...">
                        <button class="btn btn-primary px-4">Search</button>
                    </form>
                </div>
            </div>
        </div>
    </div>

    <div class="container mt-5">
        <div class="d-flex justify-content-between align-items-center mb-4">
            <h4 class="fw-bold">Available Collection</h4>
            <span class="text-secondary small">Total Results: {{ book_list|length }}</span>
        </div>
    {% endif %}

        <div class="row">
            {% for book in book_list %}
            <div class="col-md-4 mb-4">
                <div class="card h-100 shadow-lg">
                    <div class="card-body">
                        <div class="d-flex justify-content-between mb-2">
                            <small class="text-info">#LIB{{ book.id }}</small>
                            <span class="badge {{ 'badge-available' if book.status == 'Available' else 'badge-issued' }}">
                                {{ book.status }}
                            </span>
                        </div>
                        <h5 class="card-title">{{ book.title }}</h5>
                        <p class="text-secondary small mb-3">{{ book.category }} Department</p>
                        {% if view == 'admin' %}
                            <a href="/admin/delete/{{ book.id }}" class="btn btn-sm btn-outline-danger w-100">Delete Entry</a>
                        {% else %}
                            <button class="btn btn-sm btn-outline-info w-100">Book Details</button>
                        {% endif %}
                    </div>
                </div>
            </div>
            {% endfor %}
        </div>
    </div>

    <div class="modal fade" id="loginModal" tabindex="-1">
        <div class="modal-dialog modal-dialog-centered">
            <div class="modal-content border-0" style="background-color: var(--card-dark);">
                <div class="modal-body p-5 text-center">
                    <h3 class="mb-4">Student Login</h3>
                    <input type="text" class="form-control mb-3" placeholder="College ID">
                    <input type="password" class="form-control mb-3" placeholder="Password">
                    <button class="btn btn-primary w-100 mb-3">Login</button>
                    <a href="#" class="text-info small">Forgot Password?</a>
                </div>
            </div>
        </div>
    </div>

    <footer class="text-center mt-5">
        <p>© 2025 TGPCET Smart Library | Designed for Excellence</p>
    </footer>

    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
</body>
</html>
"""

@app.route('/')
def home():
    return render_template_string(DARK_TEMPLATE, book_list=books, view='student')

@app.route('/admin')
def admin_view():
    return render_template_string(DARK_TEMPLATE, book_list=books, view='admin')

@app.route('/search')
def search():
    q = request.args.get('q', '').lower()
    res = [b for b in books if q in b['title'].lower() or q in b['category'].lower()]
    return render_template_string(DARK_TEMPLATE, book_list=res, view='student')

@app.route('/admin/add', methods=['POST'])
def add():
    new_id = books[-1]['id'] + 1 if books else 1
    books.append({"id": new_id, "title": request.form['title'], "category": request.form['cat'], "status": "Available", "count": 1})
    return redirect('/admin')

@app.route('/admin/delete/<int:bid>')
def delete(bid):
    global books
    books = [b for b in books if b['id'] != bid]
    return redirect('/admin')

# --- RUN ---
def run():
    app.run(port=5000)

threading.Thread(target=run).start()
print("Website Live (Dark Theme) at:", eval_js("google.colab.kernel.proxyPort(5000)"))

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Website Live (Dark Theme) at: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev


In [10]:
from google.colab.output import eval_js

# This creates the base URL for your running Colab server
base_url = eval_js("google.colab.kernel.proxyPort(5000)")

print("--- TGPCET SMART LIBRARY ACCESS ---")
print(f"1. Student View: {base_url}")
print(f"2. Admin Panel:  {base_url}admin")

# IMPORTANT: If the link above gives a 404, click the Student link first,
# then manually add 'admin' to the end of the URL in your browser's address bar.

--- TGPCET SMART LIBRARY ACCESS ---
1. Student View: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev
2. Admin Panel:  https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.devadmin


In [11]:
import threading
from flask import Flask, render_template_string, request, redirect
from google.colab.output import eval_js

app = Flask(__name__)

# --- DATABASE ---
books = [
    {"id": 1, "title": "Machine Learning", "category": "IT", "status": "Available"},
    {"id": 2, "title": "Thermodynamics", "category": "Mechanical", "status": "Issued"}
]

# --- DARK THEME HTML ---
HTML_LAYOUT = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>TGPCET Smart Library</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body { background-color: #0f172a; color: #f8fafc; }
        .navbar { background-color: #1e293b; border-bottom: 1px solid #334155; }
        .card { background-color: #1e293b; border: 1px solid #334155; color: white; margin-bottom: 20px; }
        .btn-primary { background-color: #38bdf8; color: #0f172a; border: none; font-weight: bold; }
        .form-control { background-color: #0f172a; border: 1px solid #334155; color: white; }
    </style>
</head>
<body>
    <nav class="navbar navbar-dark px-4 py-3">
        <a class="navbar-brand fw-bold" href="/">TGPCET <span class="fw-light">Smart Library</span></a>
        <div>
            <a href="/" class="btn btn-sm btn-outline-info me-2">Student View</a>
            <a href="/admin" class="btn btn-sm btn-primary">Admin Panel</a>
        </div>
    </nav>

    <div class="container mt-5">
        {% if is_admin %}
            <h2 class="text-info mb-4">Admin: Manage Inventory</h2>
            <div class="card p-4">
                <form action="/admin/add" method="POST" class="row g-3">
                    <div class="col-md-6"><input name="title" class="form-control" placeholder="Book Title" required></div>
                    <div class="col-md-4"><input name="cat" class="form-control" placeholder="Category" required></div>
                    <div class="col-md-2"><button type="submit" class="btn btn-primary w-100">Add</button></div>
                </form>
            </div>
        {% else %}
            <div class="text-center py-5">
                <h1 class="display-4 fw-bold">Search the Archive</h1>
                <div class="row justify-content-center mt-4">
                    <div class="col-md-6">
                        <form action="/search" method="GET" class="d-flex">
                            <input name="q" class="form-control me-2" placeholder="Search by Branch or Title...">
                            <button class="btn btn-primary">Search</button>
                        </form>
                    </div>
                </div>
            </div>
        {% endif %}

        <div class="row mt-4">
            {% for book in book_list %}
            <div class="col-md-4">
                <div class="card p-3 shadow-sm">
                    <h5>{{ book.title }}</h5>
                    <p class="text-secondary small">{{ book.category }}</p>
                    <span class="badge {{ 'bg-success' if book.status == 'Available' else 'bg-danger' }} w-50">
                        {{ book.status }}
                    </span>
                    {% if is_admin %}
                        <hr><a href="/admin/delete/{{ book.id }}" class="btn btn-sm btn-outline-danger">Remove</a>
                    {% endif %}
                </div>
            </div>
            {% endfor %}
        </div>
    </div>
</body>
</html>
"""

# --- ROUTES ---
@app.route('/')
def home():
    return render_template_string(HTML_LAYOUT, book_list=books, is_admin=False)

@app.route('/admin')
def admin():
    return render_template_string(HTML_LAYOUT, book_list=books, is_admin=True)

@app.route('/search')
def search():
    q = request.args.get('q', '').lower()
    res = [b for b in books if q in b['title'].lower() or q in b['category'].lower()]
    return render_template_string(HTML_LAYOUT, book_list=res, is_admin=False)

@app.route('/admin/add', methods=['POST'])
def add():
    books.append({"id": len(books)+1, "title": request.form['title'], "category": request.form['cat'], "status": "Available"})
    return redirect('/admin')

@app.route('/admin/delete/<int:bid>')
def delete(bid):
    global books
    books = [b for b in books if b['id'] != bid]
    return redirect('/admin')

# --- EXECUTION ---
def run():
    app.run(port=5000)

threading.Thread(target=run).start()

# THIS IS THE FIX FOR YOUR LINKS
base_url = eval_js("google.colab.kernel.proxyPort(5000)")
print("--- ACCESS LINKS ---")
print(f"Student Portal: {base_url}")
print(f"Admin Portal:   {base_url.rstrip('/')}/admin")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


--- ACCESS LINKS ---
Student Portal: https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev
Admin Portal:   https://5000-m-s-3itmly2w708ck-d.us-east1-2.prod.colab.dev/admin
